In [140]:
import  pandas as p
import numpy as n
from sklearn.model_selection import StratifiedKFold 
from sklearn.metrics import f1_score, accuracy_score
from lightgbm import LGBMClassifier
import lightgbm as lgb
print("Step 1 ")



Step 1 


**Problem Description and Context**

- The dataset consists of 47 numerical features, labeled F01 through F47. These variables represent 
quantitative measurements captured by an embedded detection or monitoring system. Each feature 
corresponds to a specific operational parameter recorded during device activity cycles. 
The values are automatically generated by the detector and reflect the internal state, performance 
metrics, or environmental interactions of the device at a given time.

  Target Variable (Class):
  The objective is to determine the operational status of each device based on the 47 input features. 
  The classification label is represented by the variable Class, which indicates whether the device 
  is functioning normally or experiencing a fault condition.

  Class = 0:  Device operating under normal conditions.
  Class = 1 : Device exhibiting a faulty condition.

- We can identify it as a binary classification test as stated in the statement which check whether device is normal or faulty.

**Approach** 

- We used LightGBMClassifier as it performs well on tabular numerical datasets. it is also useful and efficient for large datasets.
- For model evaluation , Stratified 5 fold Cross Validation was applied to maintain class distribution in folds.
- Modle generates Out of fold Predictions(OOf) , that is used in tuning of probability threshold fo getting max F1 score.
- We also used Logistic Regression and XGBClassifier but they were giving less F1 score and accuracy.
  

In [141]:
train = p.read_csv("/kaggle/input/datasets/yuvrajkabadwal/lulu12345/ML Challenge Dataset/TRAIN.csv")
test = p.read_csv("/kaggle/input/datasets/yuvrajkabadwal/lulu12345/ML Challenge Dataset/TEST.csv")
train.head()
print("Step 2")

Step 2


**Loading of Dataset** 

- Used for loading the test and trained datasets in environment.
- Train.csv for training the dataset.
- Test.csv for predictions.

In [142]:
print(train.isnull().sum().sum(), "total missing")
print(train["Class"].value_counts())
print("Step 3")

0 total missing
Class
0    26465
1    17311
Name: count, dtype: int64
Step 3


**Data Inspection** 

- For understanding the  structure and quality of the model.
- We check if there are missing values in the training dataset.
- Then we check the distribution of CLASS variable to understand whivh sample belongs to which class.
  
  

In [143]:
train = train.fillna(train.median())
test = test.fillna(train.median())
x= train.drop("Class",axis =1)
y = train["Class"]
test_features =test.drop("ID",axis =1)
print("Step 4")


Step 4


**Preprocessing** 

- In this step , Preprocessing is done to prepare dataset for training.
- For handling missing values ,we fill median of training dataset.
- CLASS is seperated from input.
- ID was removed as it was not a feature and only used in FINAL.csv.


In [144]:

  model  = LGBMClassifier(
       objective = "binary",n_estimators = 5000,learning_rate = 0.01, num_leaves=96,max_depth = -1,
      min_child_samples =30,subsample =0.8,subsample_freq=5,colsample_bytree=0.8,reg_alpha=0.05,
      reg_lambda=0.05, random_state =42,n_jobs =-1
   ) 
            
            
            
print("Step 5")

Step 5


**Model Used**

- we used this as it is very good for tabular numerical datasets.
- It is fast and efficient for handling more features.
- It's best feature is that it can capture complex relationships between features and target variable.

**Hyperparameter(main)**

- n_estimators = 5000 (help in leearn more rounds of boosting ,we used 6000 but it was degrading accuracy )
- learning rate = 0.01 (used it for  better generalisation till it helped in accuracyand don"t affect the logloss)
- num_leaves = 96 (it helped in controllling tree comlexity more trees were making it unstable )

In [173]:
cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

oof_preds = n.zeros(len(x))
models = []

for fold, (train_idx, val_idx) in enumerate(cv.split(x, y)):
    
    print(f"\nFold {fold+1}")
    
    x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
    
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(
        x_train, y_train,
        eval_set=[(x_val, y_val)],
        eval_metric='binary_logloss',
        callbacks =[
            lgb.early_stopping(200),
            lgb.log_evaluation(0)
            
        ]
             )
    
    oof_preds[val_idx] = model.predict_proba(x_val)[:, 1]
    models.append(model)
print("Step 6")


Fold 1
[LightGBM] [Info] Number of positive: 14838, number of negative: 22684
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010741 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 37522, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.395448 -> initscore=-0.424468
[LightGBM] [Info] Start training from score -0.424468
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[3153]	valid_0's binary_logloss: 0.027324

Fold 2
[LightGBM] [Info] Number of positive: 14838, number of negative: 22684
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010037 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 37522, number of used fe

**Model Training** 

- We train the model using Stratified 5 fold cross validation.
- Dataset is divided into 5 folds keeping same class distribution.
- For each iteration , 4 folds used for training and 1 fold for validation.
- Our model is training on each fold.
- Early stopping is used to stop if validation score doesn't improve.
- Out of fold predictions are used later in model performance.

In [174]:
test_preds= n.zeros(len(test_features))
for model in models:
    test_preds +=model.predict_proba(test_features)[:,1]
test_preds/= len(models)
print("Step 7")

Step 7


- Test prediction generate through average of probability predictions.

In [176]:
oof_binary_05 = (oof_preds >= 0.5).astype(int)

print("F1(0.5):", f1_score(y, oof_binary_05))
print("A(0.5):", accuracy_score(y, oof_binary_05))

F1(0.5): 0.9870356374629382
A(0.5): 0.989811769005848


- Model performace using default probability threshold of 0.5.

In [183]:
best_f1 = 0
best_threshold = 0.5

for t in n.arange(0.01, 1, 0.001):
    preds = (oof_preds >= t).astype(int)
    score = f1_score(y, preds)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best_Threshold:", best_threshold)
print("Best_F1:", best_f1)
print("Step 8")

Best_Threshold: 0.3629999999999997
Best_F1: 0.9881235154394299
Step 8


**Threshold Selection**

- We searched for the max threshold to maximize our F1 score.
- Used range of 0.01 to 0.1 as max probability was coming 0.99999.
- Predictions are converted into class labels.
- Threshold which gave highest F1 gets selected.

In [184]:
final_test_preds =(test_preds >=best_threshold).astype(int)
print("Step 9")

Step 9


In [185]:
oof_binary_best = (oof_preds >= best_threshold).astype(int)

print("FinalF1:", f1_score(y, oof_binary_best))
print("FinalA:", accuracy_score(y, oof_binary_best))
print("Step 10")

FinalF1: 0.9881235154394299
FinalA: 0.9906341374269005
Step 10


**Final model** 

- After selecting the best threshold , predictions convert into class.
- Model performance using F1 score and Accuracy.
- These final scores were calculated using Out of fold Predictions made during cross validation.
- We also checked oof prediction  max and test prediction max which came out to be the 0.999999 and range is starting      from 0.000002 so threshold doesn't even matter that much here for this dataset.

In [195]:
submission  = p.DataFrame({"ID": test["ID"], "CLASS": final_test_preds})
submission.to_csv("FINAL.csv",index =False)
submission.head(20)
print("Last Step")
submission["CLASS"].value_counts() 

Last Step


CLASS
0    6652
1    4292
Name: count, dtype: int64

**Final File**

- Predicted class labels are combined with ID columns of the test dataset.
- The main dataframe is created using ID and CLASS.
- Final predictions saved in FINAL.csv